In [ ]:
# DB bootstrap: create database and minimal tables if missing
import os, pymysql
from dotenv import load_dotenv
load_dotenv()

host=os.getenv('DB_HOST','127.0.0.1')
port=int(os.getenv('DB_PORT','3306'))
user=os.getenv('DB_USER')
pw=os.getenv('DB_PASS') or ''
db=os.getenv('DB_NAME') or os.getenv('DB_NAME_PRIVATE')
print({'host':host,'port':port,'db':db})

# Create database if not exists
conn = pymysql.connect(host=host, port=port, user=user, password=pw)
try:
    with conn.cursor() as cur:
        cur.execute(f"CREATE DATABASE IF NOT EXISTS `{db}` CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci")
    conn.commit()
finally:
    conn.close()

# Create required tables if not exist
conn = pymysql.connect(host=host, port=port, user=user, password=pw, db=db, charset='utf8mb4', cursorclass=pymysql.cursors.DictCursor)
try:
    with conn.cursor() as cur:
        cur.execute(
            """
            CREATE TABLE IF NOT EXISTS `youtube_videos_raw` (
              `video_id` varchar(50) NOT NULL,
              `playlist_id` varchar(100) DEFAULT NULL,
              `raw_data` json DEFAULT NULL,
              `fetched_at` datetime DEFAULT CURRENT_TIMESTAMP,
              `processed` tinyint(1) DEFAULT '0',
              PRIMARY KEY (`video_id`),
              KEY `idx_yvraw_playlist` (`playlist_id`),
              KEY `idx_yvraw_processed` (`processed`)
            )
            """
        )
        cur.execute(
            """
            CREATE TABLE IF NOT EXISTS `youtube_metrics` (
              `video_id` varchar(50) NOT NULL,
              `view_count` bigint DEFAULT NULL,
              `like_count` bigint DEFAULT NULL,
              `dislike_count` bigint DEFAULT NULL,
              `comment_count` bigint DEFAULT NULL,
              `subscriber_count` bigint DEFAULT NULL,
              `metrics_date` date NOT NULL,
              `fetched_at` datetime NOT NULL DEFAULT CURRENT_TIMESTAMP,
              PRIMARY KEY (`video_id`,`metrics_date`)
            )
            """
        )
        cur.execute(
            """
            CREATE TABLE IF NOT EXISTS `youtube_etl_runs` (
              `channel_id` varchar(64) NOT NULL,
              `run_date` date NOT NULL,
              `started_at` datetime NOT NULL DEFAULT CURRENT_TIMESTAMP,
              `finished_at` datetime DEFAULT NULL,
              `status` varchar(16) NOT NULL DEFAULT 'started',
              PRIMARY KEY (`channel_id`,`run_date`)
            )
            """
        )
    conn.commit()
    with conn.cursor() as cur:
        cur.execute("SHOW TABLES LIKE 'youtube_%'")
        tables=[row[next(iter(row))] for row in cur.fetchall()]
        print({'tables': tables})
finally:
    conn.close()

# YouTube ETL: fetch channel videos and store raw + daily-max metrics

This section fetches all upload videos for configured channels using YouTube Data API v3,
stores the raw video JSON into `youtube_videos_raw`, and upserts a daily max row into `youtube_metrics`.

Set your `YOUTUBE_API_KEY` and channel vars in `.env` before running.

In [ ]:
# Imports and helpers
import os
import json
import re
from datetime import datetime, date
import requests
import pymysql
from urllib.parse import urlparse, parse_qs
from dotenv import load_dotenv

load_dotenv()
YOUTUBE_API_KEY = os.getenv('YOUTUBE_API_KEY')
DB_HOST = os.getenv('DB_HOST', '127.0.0.1')
DB_PORT = int(os.getenv('DB_PORT', '3306'))
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')
DB_NAME = os.getenv('DB_NAME')

assert YOUTUBE_API_KEY, 'YOUTUBE_API_KEY not set in .env'
assert DB_NAME, 'DB_NAME not set in .env'


def get_channel_id_from_url(url):
    # Accepts channel URLs like /channel/UC..., /user/..., or /@handle and resolves to channelId via API when needed
    url = url.strip()
    if url.endswith('/'):
        url = url[:-1]
    parsed = urlparse(url)
    path = parsed.path.lstrip('/')
    # direct channel id
    m = re.match(r'channel/(UC[0-9A-Za-z_-]{20,})', path)
    if m:
        return m.group(1)
    # handle or user or @handle
    candidate = path.split('/')[-1]
    if candidate:
        # call search.list to resolve handle/user to a channelId
        url_api = 'https://www.googleapis.com/youtube/v3/search'
        params = {'key': YOUTUBE_API_KEY, 'q': candidate, 'type': 'channel', 'part': 'snippet', 'maxResults': 5}
        r = requests.get(url_api, params=params)
        data = r.json()
        items = data.get('items', [])
        if items:
            return items[0]['snippet']['channelId']
    return None


def get_uploads_playlist_for_channel(channel_id):
    # channels.list part=contentDetails to get uploads playlist id
    url_api = 'https://www.googleapis.com/youtube/v3/channels'
    params = {'key': YOUTUBE_API_KEY, 'id': channel_id, 'part': 'contentDetails'}
    r = requests.get(url_api, params=params)
    data = r.json()
    items = data.get('items', [])
    if not items:
        return None
    return items[0]['contentDetails']['relatedPlaylists'].get('uploads')


def list_playlist_videos(playlist_id):
    # yields playlistItem per page
    url_api = 'https://www.googleapis.com/youtube/v3/playlistItems'
    params = {'key': YOUTUBE_API_KEY, 'playlistId': playlist_id, 'part': 'contentDetails,snippet', 'maxResults': 50}
    while True:
        r = requests.get(url_api, params=params)
        data = r.json()
        for it in data.get('items', []):
            yield it
        if 'nextPageToken' in data:
            params['pageToken'] = data['nextPageToken']
        else:
            break


def get_videos_details(video_ids):
    # video_ids up to 50
    url_api = 'https://www.googleapis.com/youtube/v3/videos'
    params = {'key': YOUTUBE_API_KEY, 'id': ','.join(video_ids), 'part': 'snippet,contentDetails,statistics'}
    r = requests.get(url_api, params=params)
    return r.json()


def connect_db():
    return pymysql.connect(host=DB_HOST, port=DB_PORT, user=DB_USER, password=DB_PASS, db=DB_NAME, charset='utf8mb4', cursorclass=pymysql.cursors.DictCursor)


def upsert_video_raw(conn, video_id, playlist_id, raw_json):
    with conn.cursor() as cur:
        sql = "INSERT INTO youtube_videos_raw (video_id, playlist_id, raw_data, fetched_at, processed) VALUES (%s, %s, %s, NOW(), 0) ON DUPLICATE KEY UPDATE raw_data = VALUES(raw_data), fetched_at = NOW(), processed = 0"
        cur.execute(sql, (video_id, playlist_id, json.dumps(raw_json)))
    conn.commit()


def upsert_daily_metrics(conn, video_id, view_count, like_count, comment_count, fetched_dt=None):
    # maintain one max-per-day row: if existing for (video_id, today) has lower view_count, replace it
    if fetched_dt is None:
        fetched_dt = datetime.utcnow()
    metrics_date = fetched_dt.date()
    with conn.cursor() as cur:
        # Try insert; on duplicate, update only if view_count is greater
        sql = "INSERT INTO youtube_metrics (video_id, view_count, like_count, dislike_count, comment_count, subscriber_count, metrics_date, fetched_at) VALUES (%s,%s,%s,%s,%s,NULL,%s,NOW()) ON DUPLICATE KEY UPDATE view_count = IF(VALUES(view_count) > view_count, VALUES(view_count), view_count), like_count = IF(VALUES(like_count) > like_count, VALUES(like_count), like_count), comment_count = IF(VALUES(comment_count) > comment_count, VALUES(comment_count), comment_count), fetched_at = NOW()"
        cur.execute(sql, (video_id, view_count, like_count, 0, comment_count, metrics_date))
    conn.commit()

print('Helpers loaded')

In [ ]:
# Main: fetch all uploads for a channel URL and store raw + daily metrics
def fetch_channel_uploads_and_store(channel_url, limit=None):
    channel_id = get_channel_id_from_url(channel_url)
    if not channel_id:
        raise ValueError(f'Could not resolve channel id from {channel_url}')
    uploads_pid = get_uploads_playlist_for_channel(channel_id)
    if not uploads_pid:
        raise ValueError(f'No uploads playlist for channel {channel_id}')
    conn = connect_db()
    try:
        seen = 0
        batch_ids = []
        for item in list_playlist_videos(uploads_pid):
            vid = item['contentDetails']['videoId']
            playlist_id = uploads_pid
            # fetch details in batches of 50
            batch_ids.append(vid)
            if len(batch_ids) >= 50:
                details = get_videos_details(batch_ids)
                for v in details.get('items', []):
                    vid2 = v['id']
                    upsert_video_raw(conn, vid2, playlist_id, v)
                    stats = v.get('statistics', {})
                    view_count = int(stats.get('viewCount') or 0)
                    like_count = int(stats.get('likeCount') or 0)
                    comment_count = int(stats.get('commentCount') or 0)
                    upsert_daily_metrics(conn, vid2, view_count, like_count, comment_count)
                batch_ids = []
            seen += 1
            if limit and seen >= limit:
                break
        # final batch
        if batch_ids:
            details = get_videos_details(batch_ids)
            for v in details.get('items', []):
                vid2 = v['id']
                upsert_video_raw(conn, vid2, uploads_pid, v)
                stats = v.get('statistics', {})
                view_count = int(stats.get('viewCount') or 0)
                like_count = int(stats.get('likeCount') or 0)
                comment_count = int(stats.get('commentCount') or 0)
                upsert_daily_metrics(conn, vid2, view_count, like_count, comment_count)
    finally:
        conn.close()
    print(f'Finished fetching for {channel_url}')


# Example: run for BicFizzle (reads from .env var YT_BICFIZZLE_YT)
ch = os.getenv('YT_BICFIZZLE_YT')
if ch:
    print('Running fetch for', ch)
    fetch_channel_uploads_and_store(ch, limit=None)
else:
    print('Set YT_BICFIZZLE_YT in .env to run example')

In [ ]:
# Class-based ETL run for one channel with bulletproof runner
import os
from functools import partial
from web.etl_entrypoints import run_channel_etl
from web.bulletproof_runner import run_cell_bulletproof

channel_url = os.getenv('YT_BICFIZZLE_YT')

# Use a picklable top-level function with functools.partial instead of a lambda
fn = partial(run_channel_etl, channel_url, None)
res = run_cell_bulletproof(fn, timeout_s=600, mem_mb=1024)

if res.status != 'success':
    print('ETL failed:', res.status, res.error_message)
else:
    s = res.result
    print({'channel_url': s.channel_url, 'channel_id': s.channel_id, 'uploads_playlist_id': s.uploads_playlist_id, 'videos_seen': s.videos_seen, 'raw_upserts': s.raw_upserts, 'metrics_upserts': s.metrics_upserts, 'errors': s.errors})

## Run for all configured artist channels

We will iterate over all channel URLs in `.env` (BicFizzle, Cobrah, Corook, Enchanting, Flyana Boss), running the ETL and printing a summary per channel. This batches raw into `youtube_videos_raw` first, then upserts daily-max metrics.

In [ ]:
# Multi-channel run (auto-discover YT_* env vars)
import os
from functools import partial
from web.etl_entrypoints import run_channel_etl
from web.bulletproof_runner import run_cell_bulletproof

# Discover any env vars whose names start with YT_ and values look like YouTube URLs
channels = []
for k, v in os.environ.items():
    if not k.startswith('YT_'):
        continue
    if not v:
        continue
    if 'youtube.com' in v or v.startswith('http'):
        channels.append(v)

if not channels:
    print('No YT_* env vars found with YouTube URLs')

for ch in channels:
    fn = partial(run_channel_etl, ch, None)
    res = run_cell_bulletproof(fn, timeout_s=900, mem_mb=1024)
    if res.status == 'success':
        s = res.result
        print({'channel_url': s.channel_url, 'channel_id': s.channel_id, 'uploads_playlist_id': s.uploads_playlist_id, 'videos_seen': s.videos_seen, 'raw_upserts': s.raw_upserts, 'metrics_upserts': s.metrics_upserts, 'errors': s.errors})
    else:
        print('ETL failed for', ch, res.status, res.error_message)

In [ ]:
# DB sanity check (run in ETL.ipynb)
import os, pymysql
from dotenv import load_dotenv
load_dotenv()

cfg = dict(
    host=os.getenv('DB_HOST','127.0.0.1'),
    port=int(os.getenv('DB_PORT','3306')),
    user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASS'),
)
db_name = os.getenv('DB_NAME')

print({'host': cfg['host'], 'port': cfg['port'], 'db': db_name})

try:
    # connect to server first to check DB presence
    conn = pymysql.connect(db='information_schema', charset='utf8mb4', cursorclass=pymysql.cursors.DictCursor, **cfg)
    with conn.cursor() as cur:
        cur.execute("SELECT SCHEMA_NAME FROM SCHEMATA WHERE SCHEMA_NAME=%s", (db_name,))
        exists = cur.fetchone() is not None
        print({'database_exists': exists})
        if not exists:
            raise RuntimeError(f"Database {db_name} not found")

    conn.select_db(db_name)
    with conn.cursor() as cur:
        sql = (
            "SELECT TABLE_NAME AS name FROM information_schema.tables "
            "WHERE table_schema=%s AND table_name LIKE %s ORDER BY table_name"
        )
        cur.execute(sql, (db_name, 'youtube_%'))
        rows = cur.fetchall()
        tables = [r.get('name') or next(iter(r.values())) for r in rows]
        print({'tables': tables})

        counts = {}
        for t in tables:
            cur.execute(f"SELECT COUNT(*) AS c FROM `{t}`")
            counts[t] = cur.fetchone()['c']
        print({'row_counts': counts})

except Exception as e:
    print('DB_CHECK_ERROR:', type(e).__name__, str(e))
    raise
finally:
    try:
        conn.close()
    except:
        pass

# Sentiment Scoring

Run sentiment analysis on recent comments and update summaries.

In [ ]:
from web.etl_entrypoints import run_sentiment_scoring
stats = run_sentiment_scoring(batch_size=1000, loop=True, update_summary=True, snapshot_daily=True)
print(stats)
